# Implmementing a Simple Research Assistant Agent

1. Setup env
2. Setup tools/model/bind model to tools
3. Setup assistant as a simple function that invokes the model + tools
4. Initialize the state
5. Setup graph with nodes and edges
6. Compile + invoke

In [ ]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("OPENAI_API_KEY")
_set_env("TAVILY_API_KEY")

In [ ]:
# setup the tools
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=5)

search_tool.invoke({"query": "What is langgraph?"})

In [ ]:
# setup model + bind to tools
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [search_tool]
llm_with_tools = llm.bind_tools(tools)
llm_with_tools.invoke("What is langgraph?")

In [ ]:
# payload prepared by the LLM for calling the search tool
output = llm_with_tools.invoke("What is langgraph?")

output.tool_calls

In [ ]:
from langgraph.graph import MessagesState

class ResearchAssistantState(MessagesState):
    pass

In [ ]:
def assistant(state: ResearchAssistantState):
    sys_msg = "You are a helpful assistant that can search \
        the web for information and return an organized mini report containing the answer with references used."
    return {"messages":llm_with_tools.invoke([sys_msg] + state["messages"])}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image

builder = StateGraph(ResearchAssistantState)

# Nodes
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Edges 
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

Image(graph.get_graph().draw_mermaid_png())

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

output = graph.invoke({"messages": [
    HumanMessage(content="Look up live-training courses by Lucas Soares at O'Reilly Media.")]}, config=config)

for m in output["messages"]:
    m.pretty_print()

In [ ]:
message = HumanMessage(content="What did I ask you about?")

output = graph.invoke({"messages": [message]}, config=config)

for m in output["messages"]:
    m.pretty_print()